In [68]:
import pandas as pd
from binance.client import Client
client = Client()

In [ ]:
info = client.get_exchange_info()
df_pairs = pd.DataFrame(info["symbols"])
df_active = df_pairs[df_pairs["status"] == "TRADING"][
    ["symbol", "baseAsset", "quoteAsset"]
]


In [65]:
all_tickers = Client().get_all_tickers()
df_tickers = pd.DataFrame(all_tickers)
df_tickers["price"] = df_tickers["price"].astype(float)

In [ ]:
df_usdt = df_active[df_active['quoteAsset']=="USDT"]

In [33]:
df_merged = pd.merge(df_usdt, df_tickers, on="symbol", how="left")

In [ ]:
## Known altcoins close to USDT price
altcoins = ["XRP", "DOT", "AXS", "GAS", "PROM", "PENDLE", "ORCA", "VANA", "U", "ATM", "ASR", "MOVR", "CVX", "RPL", "ZRO", "EUL"]

In [67]:
stable_coins = df_merged[(df_merged["price"] < 1.3) & (df_merged["price"] > 0.8)]['baseAsset'].to_list()
stable_coins  = [i for i in stable_coins if i not in altcoins]
stable_coins

['TUSD',
 'USDC',
 'EUR',
 'USDP',
 'FDUSD',
 'AEUR',
 'EURI',
 'XUSD',
 'USD1',
 'BFUSD',
 'USDE',
 'RLUSD',
 'USDS']

In [79]:
stable_coin_df = df_active[(df_active['quoteAsset']=="USDT") & df_active["baseAsset"].isin(stable_coins)].reset_index(drop=True)
stable_coin_symbols = stable_coin_df["symbol"].to_list()
stable_coin_symbols

['TUSDUSDT',
 'USDCUSDT',
 'EURUSDT',
 'USDPUSDT',
 'FDUSDUSDT',
 'AEURUSDT',
 'EURIUSDT',
 'XUSDUSDT',
 'USD1USDT',
 'BFUSDUSDT',
 'USDEUSDT',
 'RLUSDUSDT',
 'USDSUSDT']

In [85]:
stable_coins.append("USDT")
tri_arb_assets_df = df_active[df_active["quoteAsset"].isin(stable_coins)].reset_index(drop=True)
tri_arb_assets_df

,symbol,baseAsset,quoteAsset
0,BTCUSDT,BTC,USDT
1,ETHUSDT,ETH,USDT
2,BNBUSDT,BNB,USDT
3,NEOUSDT,NEO,USDT
4,LTCUSDT,LTC,USDT
...,...,...,...
813,AIGENSYNUSDC,AIGENSYN,USDC
814,GENIUSUSDT,GENIUS,USDT
815,GENIUSUSDC,GENIUS,USDC
816,OPGUSDT,OPG,USDT


In [86]:
tri_arb_assets_df.groupby("quoteAsset")["symbol"].count()

quoteAsset
EUR       29
EURI       3
FDUSD     37
RLUSD      1
USD1      27
USDC     289
USDS       2
USDT     430
Name: symbol, dtype: int64

In [87]:
stable_coins_low_vol = [i for i in stable_coins if i not in ["USDT", "USDC"]]
tri_arb_low_vol_assets_df = df_active[df_active["quoteAsset"].isin(stable_coins_low_vol)].reset_index(drop=True)
tri_arb_low_vol_assets_df

,symbol,baseAsset,quoteAsset
0,BTCEUR,BTC,EUR
1,ETHEUR,ETH,EUR
2,BNBEUR,BNB,EUR
3,XRPEUR,XRP,EUR
4,LINKEUR,LINK,EUR
...,...,...,...
94,ETHUSDS,ETH,USDS
95,币安人生USD1,币安人生,USD1
96,CHIPUSD1,CHIP,USD1
97,XAUTUSD1,XAUT,USD1


In [132]:
all_24h_stats = client.get_ticker()
df_all = pd.DataFrame(all_24h_stats)
all_vol = df_all[["symbol", "weightedAvgPrice", "volume"]].astype({"weightedAvgPrice": float, "volume": float})

all_vol["$vol"] = all_vol["weightedAvgPrice"]*all_vol["volume"]

In [104]:
universe_wo_vol = pd.merge(tri_arb_low_vol_assets_df, all_vol, on="symbol", how="left")
universe_wo_vol

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume
0,BTCEUR,BTC,EUR,53270.13854331,379.24637000
1,ETHEUR,ETH,EUR,1417.80031938,6347.52660000
2,BNBEUR,BNB,EUR,509.81861155,1308.30500000
3,XRPEUR,XRP,EUR,0.97766198,2906377.60000000
4,LINKEUR,LINK,EUR,6.71044767,21151.76000000
...,...,...,...,...,...
94,ETHUSDS,ETH,USDS,1641.89822138,60.86470000
95,币安人生USD1,币安人生,USD1,0.70665473,18609.10000000
96,CHIPUSD1,CHIP,USD1,0.03511486,158487.00000000
97,XAUTUSD1,XAUT,USD1,4189.96737533,5.44640000


In [106]:
ex_list = df_active[(df_active["quoteAsset"]=="USDT") & (df_active["baseAsset"].isin(stable_coins_low_vol))]["symbol"].to_list()
stable_ex_df = all_vol[all_vol["symbol"].isin(ex_list)]
stable_ex_df.rename(columns={"symbol":"ex_symbol", "weightedAvgPrice": "usdtPrice", "volume": "ex_volume"}, inplace=True)
stable_ex_df["quoteAsset"] = stable_ex_df["ex_symbol"].str[:-4]
stable_ex_df.reset_index(drop=True, inplace=True)

stable_ex_df

,ex_symbol,usdtPrice,ex_volume,quoteAsset
0,TUSDUSDT,0.99996630,70989.00000000,TUSD
1,EURUSDT,1.15565885,21052871.20000000,EUR
2,USDPUSDT,1.00016173,6094.00000000,USDP
3,FDUSDUSDT,0.99821322,22088829.00000000,FDUSD
4,AEURUSDT,1.14765396,1803.90000000,AEUR
5,EURIUSDT,1.15518121,1581153.60000000,EURI
6,XUSDUSDT,1.00081416,3036456.00000000,XUSD
7,USD1USDT,0.99945785,153173620.00000000,USD1
8,BFUSDUSDT,0.99937058,8390380.00000000,BFUSD
9,USDEUSDT,0.99978209,4554408.00000000,USDE


In [ ]:
universe = pd.merge(universe_wo_vol, stable_ex_df, on="quoteAsset", how="left")
universe["weightedAvgPrice"] = universe["weightedAvgPrice"].astype(float)
universe["volume"] = universe["volume"].astype(float)

universe["volume$"] = universe["weightedAvgPrice"] * universe["volume"] 


universe = universe[universe["volume$"]>100000]
universe.index.name = "no."
sorted_universe = universe.sort_values(by="weightedAvgPrice", ascending=False).reset_index(drop=True)
sorted_universe

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume,ex_symbol,usdtPrice,ex_volume,volume$
0,BTCFDUSD,BTC,FDUSD,61769.458306,1.099258e+03,FDUSDUSDT,0.99821322,22088829.00000000,6.790058e+07
1,BTCUSD1,BTC,USD1,61595.376881,2.131264e+03,USD1USDT,0.99945785,153173620.00000000,1.312760e+08
2,BTCEUR,BTC,EUR,53270.138543,3.792464e+02,EURUSDT,1.15565885,21052871.20000000,2.020251e+07
3,BTCEURI,BTC,EURI,53267.095531,2.018750e+00,EURIUSDT,1.15518121,1581153.60000000,1.075329e+05
4,ETHUSD1,ETH,USD1,1640.075863,3.765146e+04,USD1USDT,0.99945785,153173620.00000000,6.175125e+07
5,ETHFDUSD,ETH,FDUSD,1637.734287,1.101101e+04,FDUSDUSDT,0.99821322,22088829.00000000,1.803311e+07
6,ETHEUR,ETH,EUR,1417.800319,6.347527e+03,EURUSDT,1.15565885,21052871.20000000,8.999525e+06
7,BNBFDUSD,BNB,FDUSD,591.552667,4.488850e+03,FDUSDUSDT,0.99821322,22088829.00000000,2.655391e+06
8,BNBUSD1,BNB,USD1,589.786851,4.242357e+04,USD1USDT,0.99945785,153173620.00000000,2.502087e+07
9,BNBEUR,BNB,EUR,509.818612,1.308305e+03,EURUSDT,1.15565885,21052871.20000000,6.669982e+05


In [140]:
altcoins = ["BTC", "SOL", "ETH", "BNB", "ZEC", "XRP","SUI", "LINK"]
altcoin_symbols = [i+"USDT" for i in altcoins]
altcoin_symbols_df = all_vol[all_vol["symbol"].isin(altcoin_symbols)]
altcoin_symbols_df.rename(columns={"symbol":"usdt_ob", "weightedAvgPrice": "usdtAssetPrice", "volume": "ob2_volume"}, inplace=True)

altcoin_symbols_df["baseAsset"] = altcoin_symbols_df["usdt_ob"].str.replace("USDT", "", regex=False)

altcoin_symbols_df

,usdt_ob,usdtAssetPrice,ob2_volume,$vol,baseAsset
11,BTCUSDT,61525.331661,1.711729e+04,1.053147e+09,BTC
12,ETHUSDT,1636.995268,3.038689e+05,4.974319e+08,ETH
98,BNBUSDT,589.005584,1.168693e+05,6.883668e+07,BNB
306,XRPUSDT,1.124080,1.107878e+08,1.245343e+08,XRP
431,LINKUSDT,7.759522,2.038327e+06,1.581644e+07,LINK
477,ZECUSDT,435.996131,5.049197e+05,2.201430e+08,ZEC
779,SOLUSDT,64.462465,2.856984e+06,1.841682e+08,SOL
2205,SUIUSDT,0.746522,3.876759e+07,2.894088e+07,SUI


In [142]:
universe_w_tri_asset = pd.merge(sorted_universe, altcoin_symbols_df, on="baseAsset", how="left")
universe_w_tri_asset

,symbol,baseAsset,quoteAsset,weightedAvgPrice,volume,ex_symbol,usdtPrice,ex_volume,volume$,usdt_ob,usdtAssetPrice,ob2_volume,$vol
0,BTCFDUSD,BTC,FDUSD,61769.458306,1.099258e+03,FDUSDUSDT,0.99821322,22088829.00000000,6.790058e+07,BTCUSDT,61525.331661,1.711729e+04,1.053147e+09
1,BTCUSD1,BTC,USD1,61595.376881,2.131264e+03,USD1USDT,0.99945785,153173620.00000000,1.312760e+08,BTCUSDT,61525.331661,1.711729e+04,1.053147e+09
2,BTCEUR,BTC,EUR,53270.138543,3.792464e+02,EURUSDT,1.15565885,21052871.20000000,2.020251e+07,BTCUSDT,61525.331661,1.711729e+04,1.053147e+09
3,BTCEURI,BTC,EURI,53267.095531,2.018750e+00,EURIUSDT,1.15518121,1581153.60000000,1.075329e+05,BTCUSDT,61525.331661,1.711729e+04,1.053147e+09
4,ETHUSD1,ETH,USD1,1640.075863,3.765146e+04,USD1USDT,0.99945785,153173620.00000000,6.175125e+07,ETHUSDT,1636.995268,3.038689e+05,4.974319e+08
5,ETHFDUSD,ETH,FDUSD,1637.734287,1.101101e+04,FDUSDUSDT,0.99821322,22088829.00000000,1.803311e+07,ETHUSDT,1636.995268,3.038689e+05,4.974319e+08
6,ETHEUR,ETH,EUR,1417.800319,6.347527e+03,EURUSDT,1.15565885,21052871.20000000,8.999525e+06,ETHUSDT,1636.995268,3.038689e+05,4.974319e+08
7,BNBFDUSD,BNB,FDUSD,591.552667,4.488850e+03,FDUSDUSDT,0.99821322,22088829.00000000,2.655391e+06,BNBUSDT,589.005584,1.168693e+05,6.883668e+07
8,BNBUSD1,BNB,USD1,589.786851,4.242357e+04,USD1USDT,0.99945785,153173620.00000000,2.502087e+07,BNBUSDT,589.005584,1.168693e+05,6.883668e+07
9,BNBEUR,BNB,EUR,509.818612,1.308305e+03,EURUSDT,1.15565885,21052871.20000000,6.669982e+05,BNBUSDT,589.005584,1.168693e+05,6.883668e+07


In [151]:
signal = ['symbol', 'baseAsset', 'weightedAvgPrice', 'usdtPrice' , 'volume$', 'usdt_ob','usdtAssetPrice', '$vol']
arb_in_universe = universe_w_tri_asset[signal].dropna()

arb_in_universe = arb_in_universe.astype({"weightedAvgPrice": float, "usdtPrice": float})

arb_in_universe["implied_usdt_price"] = arb_in_universe["weightedAvgPrice"] * arb_in_universe["usdtPrice"]

arb_in_universe

,symbol,baseAsset,weightedAvgPrice,usdtPrice,volume$,usdt_ob,usdtAssetPrice,$vol,implied_usdt_price
0,BTCFDUSD,BTC,61769.458306,0.998213,6.790058e+07,BTCUSDT,61525.331661,1.053147e+09,61659.089873
1,BTCUSD1,BTC,61595.376881,0.999458,1.312760e+08,BTCUSDT,61525.331661,1.053147e+09,61561.982947
2,BTCEUR,BTC,53270.138543,1.155659,2.020251e+07,BTCUSDT,61525.331661,1.053147e+09,61562.107048
3,BTCEURI,BTC,53267.095531,1.155181,1.075329e+05,BTCUSDT,61525.331661,1.053147e+09,61533.147869
4,ETHUSD1,ETH,1640.075863,0.999458,6.175125e+07,ETHUSDT,1636.995268,4.974319e+08,1639.186696
5,ETHFDUSD,ETH,1637.734287,0.998213,1.803311e+07,ETHUSDT,1636.995268,4.974319e+08,1634.808016
6,ETHEUR,ETH,1417.800319,1.155659,8.999525e+06,ETHUSDT,1636.995268,4.974319e+08,1638.493487
7,BNBFDUSD,BNB,591.552667,0.998213,2.655391e+06,BNBUSDT,589.005584,6.883668e+07,590.495693
8,BNBUSD1,BNB,589.786851,0.999458,2.502087e+07,BNBUSDT,589.005584,6.883668e+07,589.467098
9,BNBEUR,BNB,509.818612,1.155659,6.669982e+05,BNBUSDT,589.005584,6.883668e+07,589.176390


In [156]:
print(altcoin_symbols_df['usdt_ob'].tolist())

['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'LINKUSDT', 'ZECUSDT', 'SOLUSDT', 'SUIUSDT']


In [163]:
arb_in_universe['alpha'] = abs((arb_in_universe['usdtAssetPrice'] - arb_in_universe['implied_usdt_price'])/arb_in_universe['usdtAssetPrice'])

arb_in_universe.sort_values(by='alpha', ascending=False).reset_index(drop=True)

,symbol,baseAsset,weightedAvgPrice,usdtPrice,volume$,usdt_ob,usdtAssetPrice,$vol,implied_usdt_price,alpha
0,ZECUSD1,ZEC,443.392904,0.999458,3.170388e+05,ZECUSDT,435.996131,2.201430e+08,443.152518,0.016414
1,XRPUSD1,XRP,1.132177,0.999458,6.748563e+06,XRPUSDT,1.124080,1.245343e+08,1.131563,0.006657
2,XRPEUR,XRP,0.977662,1.155659,2.841455e+06,XRPUSDT,1.124080,1.245343e+08,1.129844,0.005127
3,SUIFDUSD,SUI,0.744076,0.998213,3.575139e+05,SUIUSDT,0.746522,2.894088e+07,0.742746,0.005059
4,SOLUSD1,SOL,64.789589,0.999458,1.971388e+07,SOLUSDT,64.462465,1.841682e+08,64.754463,0.004530
5,XRPFDUSD,XRP,1.130090,0.998213,1.572054e+06,XRPUSDT,1.124080,1.245343e+08,1.128070,0.003550
6,XRPRLUSD,XRP,1.127032,1.000741,1.144203e+06,XRPUSDT,1.124080,1.245343e+08,1.127868,0.003369
7,SUIEUR,SUI,0.644242,1.155659,1.360802e+05,SUIUSDT,0.746522,2.894088e+07,0.744524,0.002677
8,SOLFDUSD,SOL,64.743643,0.998213,5.955552e+06,SOLUSDT,64.462465,1.841682e+08,64.627960,0.002567
9,BNBFDUSD,BNB,591.552667,0.998213,2.655391e+06,BNBUSDT,589.005584,6.883668e+07,590.495693,0.002530


In [166]:
print(arb_in_universe.sort_values(by='alpha', ascending=False).reset_index(drop=True)["symbol"].to_list())

['ZECUSD1', 'XRPUSD1', 'XRPEUR', 'SUIFDUSD', 'SOLUSD1', 'XRPFDUSD', 'XRPRLUSD', 'SUIEUR', 'SOLFDUSD', 'BNBFDUSD', 'BTCFDUSD', 'SUIUSD1', 'ETHUSD1', 'ETHFDUSD', 'SOLEUR', 'ETHEUR', 'BNBUSD1', 'BTCEUR', 'BTCUSD1', 'LINKEUR', 'BNBEUR', 'BTCEURI']


In [171]:
print(arb_in_universe["usdt_ob"].unique())

<ArrowStringArray>
[ 'BTCUSDT',  'ETHUSDT',  'BNBUSDT',  'ZECUSDT',  'SOLUSDT', 'LINKUSDT',
  'XRPUSDT',  'SUIUSDT']
Length: 8, dtype: str
